# 08 — Production: Meta Tooling

**Stage 8 of the workshop.** An agent that writes and hot-loads its own tool at runtime, then invokes it — with a deterministic fallback so the demo doesn't stall on small-model unreliability.

## Problem

Getting from "works on my laptop" to something a team can rely on — this script is a fun edge case of that theme: an agent extending its own capabilities at runtime rather than relying only on tools you hand-wrote ahead of time.

## Concept

The agent is given exactly two real tools — `editor` (to write a new tool's source file) and `load_tool` (to hot-register it) — and instructed to *actually call* them rather than narrate JSON. This is meta-tooling: the agent's own tool call writes and loads new capability into itself mid-run.

Same architectural point as Module 5: `Agent` doesn't change when what it can do changes — tools are just more tools, even when the agent authored them itself a moment ago.

## Architecture

```
User: "Create a tool called char_counter ..., then load it."
        │
        ▼
  agent (tools=[load_tool, editor])
        │
        ├──tool call──▶ editor(create, tools/char_counter.py)
        │                    writes TOOL_SPEC + char_counter(tool, **kwargs)
        │
        └──tool call──▶ load_tool(tools/char_counter.py)
                             registers char_counter as a live tool
        │
        ▼
  [validation] does tools/char_counter.py exist and look right?
        │
     no ──▶ write FALLBACK_TOOL_SOURCE instead, load_tool() it
        │
        ▼
  agent.tool.char_counter(text="hello world")  ── direct invocation
```

## Step 1 — Environment and model setup

`BYPASS_TOOL_CONSENT` must be set before `strands_tools` import. `strands_tools.shell` imports POSIX-only `pty`/`termios`, unavailable on Windows — dropped here; `editor` + `load_tool` alone cover the meta-tooling pattern.

In [1]:
import os
import sys
from pathlib import Path

os.environ.setdefault("BYPASS_TOOL_CONSENT", "true")  # skip interactive y/n prompt (breaks under this shell)
sys.path.insert(0, str(Path.cwd().parent))

from model_provider import get_model
from strands import Agent
from strands_tools import editor, load_tool

# NOTE: strands_tools.shell imports POSIX-only `pty`/`termios`, unavailable
# on Windows. Dropped here — editor + load_tool alone cover the meta-tooling
# pattern (write tool file, hot-load it, invoke it).

model = get_model()


## Step 2 — The tool-building agent

In [2]:
TOOL_BUILDER_SYSTEM_PROMPT = """You are a tool-building agent with access to two
real tools: editor and load_tool. You must actually CALL these tools — never
describe or print what a tool call would look like as text or JSON.

When asked to create a tool named <tool_name>:
1. Call the editor tool (command="create") to write tools/<tool_name>.py. The
   file must define a TOOL_SPEC dict (name, description, inputSchema) and a
   function <tool_name>(tool, **kwargs) that reads its argument from
   tool["input"]["<param>"] (NOT from kwargs) and returns a ToolResult dict
   with keys toolUseId, status, content.
2. Call the load_tool tool to register tools/<tool_name>.py.

Do both as actual tool calls. Do not narrate the JSON — invoke the tools."""

agent = Agent(model=model, system_prompt=TOOL_BUILDER_SYSTEM_PROMPT, tools=[load_tool, editor])

## Step 3 — Deterministic fallback tool source

Small local models sometimes narrate tool calls as text instead of invoking them, or mangle a 3rd chained call. This known-good fallback source keeps the hot-load/invoke half of the demo deterministic even if the agent-authored version comes out malformed.

In [3]:
# ponytail: qwen2.5:7b is unreliable formatting multi-field JSON tool calls
# several steps into a chain, so the agent-authored tools/char_counter.py may
# come out malformed. Fall back to a known-good file so the hot-load/invoke
# half of the demo still runs deterministically. Upgrade path: drop this once
# testing a model with sturdier tool-calling (e.g. qwen2.5:14b+, llama3.1).
FALLBACK_TOOL_SOURCE = '''from typing import Any
from strands.types.tools import ToolResult, ToolUse

TOOL_SPEC = {
    "name": "char_counter",
    "description": "Counts characters in a text string.",
    "inputSchema": {
        "json": {
            "type": "object",
            "properties": {"text": {"type": "string", "description": "Text to count."}},
            "required": ["text"],
        }
    },
}


def char_counter(tool: ToolUse, **kwargs: Any) -> ToolResult:
    text = tool["input"]["text"]
    return {
        "toolUseId": tool["toolUseId"],
        "status": "success",
        "content": [{"text": f"Character count: {len(text)}"}],
    }
'''


## Step 4 — Run it: ask the agent to build and load the tool

In [4]:
result = agent(
    "Create a tool called char_counter that counts characters in a text "
    "string, then load it."
)
print(result)


DEPRECATION WARNING: editor is deprecated. This warning becomes an error log in v0.9.0. Migration path: use the file_editor tool vended by strands-agents (from strands.vended_tools import file_editor).



Tool #1: editor

Tool #2: load_tool
I've created the `char_counter` tool and loaded it successfully. The tool counts characters in a text string, as specified in the `char_counter` function's description.I've created the `char_counter` tool and loaded it successfully. The tool counts characters in a text string, as specified in the `char_counter` function's description.



## Step 5 — Validate, fall back if needed, then invoke directly

In [5]:
tool_path = "tools/char_counter.py"
valid = False
if os.path.exists(tool_path):
    with open(tool_path) as f:
        valid = "text" in f.read()
if not valid:
    os.makedirs("tools", exist_ok=True)
    with open(tool_path, "w") as f:
        f.write(FALLBACK_TOOL_SOURCE)
    agent.tool.load_tool(path=tool_path, name="char_counter")

direct = agent.tool.char_counter(text="hello world")
print(direct)


{'toolUseId': 'tooluse_char_counter_418012763', 'status': 'success', 'content': [{'text': "Character count for 'hello world': 11"}]}


## Failure mode to know about

Small local models sometimes narrate tool calls as text instead of invoking them, or mangle a 3rd chained call. The script has a deterministic fallback built in (Step 3/5 above) so the demo doesn't stall — don't remove it.